In [0]:
from sdds.common.util import NotebookUtil

# COMMAND ----------
# Databricks position: 2.0
import time
import uuid
from datetime import datetime, timedelta

from pyspark.sql.functions import (
    col, lit, when, sum as spark_sum, max as spark_max, regexp_replace, coalesce,
    array, array_contains, sort_array, from_json, expr, round as spark_round,
)
from pyspark.sql.types import ArrayType, StringType
from pyspark import StorageLevel

# PERF: nothing in this notebook was cached before, so the same ~270-column,
# 8M-row, multi-join lineage below (source reads -> bronze subquery -> join ->
# ~40 array normalizations per side -> join -> 275 status columns) was being
# recomputed from scratch by every single .count()/.collect()/.write() call
# further down (there were 8-10 of them). That recomputation, not any single
# step, is almost certainly why this ran 20+ hours. The fix is to persist the
# expensive checkpoints once and reuse them.
CACHE_LEVEL = StorageLevel.DISK_ONLY  # wide (270+ col) + array-heavy frame; MEMORY_AND_DISK risks eviction/spill churn on serverless

# Attribute keys to ignore when comparing the `attributes` map. Every entry
# in the array<map<string,string>> whose key (the part before "->") matches
# one of these is dropped before comparison, so drift on these keys is not
# counted as a mismatch and they don't contribute to MATCH/MISSING/EXTRA
# either.
IGNORE_ATTRIBUTE_KEYS = [
    "attribute_id",
    "X_BazaarVoice_a_count_FNS", "X_BazaarVoice_count_ovr_FNS",
    "X_BazaarVoice_q_count_FNS", "X_BazaarVoice_ratings_ovr_FNS",
    "fieldandstreamofferprice",
    "4024", "4027", "4128", "4090", "2329", "4009", "4010", "4014", "4022",
    "4025", "4031", "4032", "4033", "4035", "4036", "4037", "4038", "4043",
    "4044", "4045", "4046", "4047", "4052", "4136", "4125", "4127", "4126",
    "4055", "4061", "4062", "4067", "4068", "4070", "4074", "4075", "4081",
    "4082", "4087", "4088", "4092", "4093", "4096", "4007", "4099", "4100",
    "4104", "4107", "4109", "4113", "4115", "2162", "2561", "3122", "3120",
    "3121", "2538", "1107", "3035", "1703", "4169", "4183", "4233", "4262",
    "4422", "4420", "4421", "4495", "4442", "4443", "4444", "4551", "4552",
    "PRIMARY_CATEGORY_FNS", "X_BOPIS_FS", "X_ISA_FS",
    "4724", "4691", "4906", "4949", "5005", "5071", "3444",
]
_IGNORE_KEYS_SQL_ARRAY = "array(" + ",".join(f"'{k}'" for k in IGNORE_ATTRIBUTE_KEYS) + ")"


catalog_name = NotebookUtil.notebook_param("sdds_catalog")
silver_schema_name = NotebookUtil.notebook_param("sdds_silver_schema")
gold_schema_name = NotebookUtil.notebook_param("sdds_gold_schema")
bronze_schema_name = NotebookUtil.notebook_param("sdds_bronze_schema")

# Anchor = full periodic reload (treated as source of truth), Compare = incremental stream
anchor_table_name = "catalog-stream-dbx-silver"
compare_table_name = "catalog-load-dbx-silver"
compare_inv_table_name = "catalog-load-sku"
unpub_table_name = " "
del_cat_table_name = ""

anchor_table = f"`{catalog_name}`.`{silver_schema_name}`.`{anchor_table_name}`"
compare_table = f"`{catalog_name}`.`{silver_schema_name}`.`{compare_table_name}`"

compare_sku_table = f"`{catalog_name}`.`{silver_schema_name}`.`{compare_inv_table_name}`"

umpub_cat_table = f"`{catalog_name}`.`{silver_schema_name}`.`{compare_inv_table_name}`"
del_cat_table = f"`{catalog_name}`.`{silver_schema_name}`.`{compare_inv_table_name}`"

detail_table = f"`{catalog_name}`.`{gold_schema_name}`.`field_level_comparison`"
summary_table = f"`{catalog_name}`.`{gold_schema_name}`.`field_level_comparison_summary`"
metadata_table = f"`{catalog_name}`.`{gold_schema_name}`.`comparison_run_metadata`"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{gold_schema_name}`")

# COMMAND ----------
# Databricks position: 3.0
run_id = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
run_timestamp = datetime.now()
run_date = run_timestamp.date()
#from datetime import date
#run_date= date.fromisoformat("2026-09-04")  # datetime.date(2026, 8, 10)


run_user = spark.sql("SELECT current_user()").collect()[0][0]
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
except Exception:
    notebook_path = "interactive"

print(f"Starting comparison run: {run_id}")
print(f"User: {run_user} | Notebook: {notebook_path}")

start_time = time.perf_counter()


# COMMAND ----------
# Databricks position: 4.0
df_anchor = spark.sql(f"""select * from  {anchor_table} as l
                            where l.extraction_date = '{run_date}' and type = 'sku'
                             """).drop('ggPriceIndicators_dealsPercentage','ggPriceIndicators_mapPriceIndicator','ggPriceIndicators_priceIndicator')

df_compare = spark.sql(f""" select *
                            from {compare_table}
                            where extraction_date = '{run_date}'
                            and type = 'sku'
                            and parentPartnumber IN (select distinct parentPartnumber
                            from `{catalog_name}`.`{silver_schema_name}`.`catalog-load-sku-inventory`
                            where extraction_date = '{run_date}')
                            """ ).drop('ggPriceIndicators_dealsPercentage','ggPriceIndicators_mapPriceIndicator','ggPriceIndicators_priceIndicator')

# PERF: precompute the cutoff date in Python and inline it as a literal.
# `CURRENT_DATE - INTERVAL '1' DAY` inside the SQL text is evaluated at query
# time and can prevent the optimizer from statically pushing the predicate
# down into partition pruning on some table/catalog versions -- an explicit
# date literal is guaranteed to prune partitions.
one_day_ago = (run_timestamp - timedelta(days=1)).date()

df_del_cats = spark.sql(f"""
                              /* Simplified LeafCategories Filtered Comparison */
                                WITH excluded_categories AS (
                                    SELECT DISTINCT CAST(category_id AS STRING) as category_id                                                                                                                                                                 
                                    FROM {catalog_name}.{bronze_schema_name}.deleted_category_events
                                    WHERE extraction_date >= '{one_day_ago}'
                                    UNION
                                    SELECT DISTINCT CAST(category_id AS STRING) as category_id
                                    FROM {catalog_name}.{bronze_schema_name}.unpublished_category_events
                                    WHERE extraction_date >= '{one_day_ago}'
                                ),
                                excluded_array AS (
                                    SELECT collect_set(category_id) AS excluded_ids
                                    FROM excluded_categories
                                ),
                                parsed_data AS (
                                    SELECT
                                    partnumber,
                                    split(regexp_replace(leafCategories_anchor_value, '^\\[|\\]$', ''), ',\\s*') AS anchor_array,
                                    split(regexp_replace(leafCategories_compare_value, '^\\[|\\]$', ''), ',\\s*') AS compare_array
                                    FROM {catalog_name}.{gold_schema_name}.field_level_comparison_flat
                                    WHERE type_load = 'sku' AND type_stream = 'sku'
                                    AND extraction_date >= CURRENT_DATE - INTERVAL '1' DAY
                                    AND leafCategories_anchor_value IS NOT NULL
                                    AND leafCategories_compare_value IS NOT NULL
                                    -- NO LIMIT - full dataset
                                ),
                                normalized_data AS (
                                    SELECT
                                    partnumber,
                                    array_sort(
                                        array_distinct(
                                        filter(
                                            transform(anchor_array, x -> regexp_extract(x, '_(.*)$', 1)),
                                            cat_id -> cat_id != '' AND NOT array_contains(ea.excluded_ids, cat_id)
                                        )
                                        )
                                    ) AS anchor_cats,
                                    array_sort(
                                        array_distinct(
                                        filter(
                                            transform(compare_array, x -> regexp_extract(x, '_(.*)$', 1)),
                                            cat_id -> cat_id != '' AND NOT array_contains(ea.excluded_ids, cat_id)
                                        )
                                        )
                                    ) AS compare_cats
                                    FROM parsed_data
                                    CROSS JOIN excluded_array ea
                                )

                                SELECT * FROM normalized_data
                              """)


# COMMAND ----------
# Databricks position: 4.25
df_compare.createOrReplaceTempView("vw_compare")
df_del_cats.createOrReplaceTempView("vw_del_unpub_cat")

df_final = spark.sql("""
    WITH partnumbers_to_consider AS (
        SELECT c.partnumber
        FROM vw_compare c
        INNER JOIN vw_del_unpub_cat m
          ON c.partnumber = m.partnumber
        WHERE arrays_overlap(
          transform(c.leafCategories, cat -> regexp_replace(regexp_extract(cat, '_(.*)$', 1), '\\]$', '')),
          m.compare_cats
        )
    )
    SELECT c.*
    FROM vw_compare c
    LEFT SEMI JOIN partnumbers_to_consider e
      ON c.partnumber = e.partnumber
""")
df_compare = df_final

# PERF: df_anchor and df_compare are each the root of a lineage that includes
# the base table reads, filters, and (for df_compare) the join above. From
# here on both are read multiple times (normalization, counts) -- persist
# once so those reads don't repeat the SQL parse/scan/join every time.
df_anchor = df_anchor.persist(CACHE_LEVEL)
df_compare = df_compare.persist(CACHE_LEVEL)

anchor_count = df_anchor.count()
compare_count = df_compare.count()

print(f"Anchor ({anchor_table_name}) records:  {anchor_count:,} (extraction_date={run_date})")
print(f"Compare ({compare_table_name}) records: {compare_count:,} (extraction_date={run_date})")

# COMMAND ----------
# Databricks position: 8.0
from pyspark.sql.functions import (
    col, lit, when, sum as spark_sum, avg as spark_avg, max as spark_max, regexp_replace, coalesce,
    array, array_contains, sort_array, from_json, expr, round as spark_round,
)
from pyspark.sql.types import (
    ArrayType, StringType, StructType, StructField, MapType,
    LongType, DoubleType, TimestampType, DateType,
)

# ── Array/set-valued fields (category-style, subset match) ─────────────────
# NOTE: leafCategories/parentCatgroup* are "Name_ID" formatted (suffix after the
# last underscore is the real ID) so they need the ID-extraction regex below.
# dsgCatgroups/ggCatgroups are PLAIN category-name strings with no ID suffix
# ("Bike Racks & Storage", "Outdoor", ...) -- running them through the ID-suffix
# regex turned every value into "" (no match), silently collapsing both sides
# to an empty array and producing a constant false MISMATCH. Moved to
# PLAIN_SET_FIELDS so they're compared as plain string sets instead.
CAT_SET_FIELDS = [
   "leafCategories", "plCatgroups",
    "parentCatgroup", "parentCatgroup0", "parentCatgroup1", "parentCatgroup2",
    "parentCatgroup3", "parentCatgroup4", "parentCatgroup5", "parentCatgroup6",
    "parentCatgroup7", "parentCatgroup8", "parentCatgroup9",
]
PLAIN_SET_FIELDS = ["catalogIds", "dsgCatgroups", "ggCatgroups"]

# Scalar fields whose anchor/compare types can drift (bigint vs string) --
# compare on a string cast so a type-only difference doesn't read as MISMATCH.
CAST_STRING_FIELDS = ["catentryId"]

# Long free-text/HTML fields -- compare with whitespace collapsed so
# formatting-only differences (line breaks, indentation) don't count as drift.
LONG_TEXT_FIELDS = ["longDescription", "dsgOverrides_longdescription"]

# Image/asset identifier fields -- compare case-insensitively and stripped of
# any path/URL prefix or file extension, since the same asset can be recorded
# as a bare code on one side and a full path/URL on the other.
IMAGE_FIELDS = ["fullImage"]

# ── Previously-skipped complex fields, now handled explicitly ──────────────
COMPLEX_STRUCT_FIELDS = ["seo"]                      # plain struct -> direct eqNullSafe, no norm needed
COMPLEX_ARRAY_STRUCT_FIELDS = [                        # array<struct>, no maps -> order-insensitive exact match
     "defAttributes", "floatFacets", "stringFacets", "numberFacets",
]
# array<struct{key,value,storeId,...}> fields that must be compared by
# (key, storeId), not as an opaque sorted array -- see normalize_keyed_struct_to_map.
KEYED_STRUCT_FIELDS = ["customSkuAttributes", "catgroupSeq"]

# Best-effort shapes to fall back to if a KEYED_STRUCT_FIELDS column ever
# shows up as a raw JSON string on one side instead of an already-parsed
# array<struct<...>> (schema drift between independently-maintained
# bronze->silver jobs). Only used by normalize_keyed_struct_to_map when
# df.schema[field] is StringType; a native array<struct> column never touches
# this and keeps reading its real per-DataFrame field list as before.
KEYED_STRUCT_JSON_SCHEMA = {
    "customSkuAttributes": "array<struct<key:string,value:string,storeId:string>>",
    "catgroupSeq": "array<struct<key:string,seq:string>>",
}
COMPLEX_ARRAY_MAP_FIELDS = ["attributes"]              # array<map<string,string>> -> convert maps to sortable structs first

def _array_source(df, field):
    return f"from_json({field}, 'array<string>')" if isinstance(df.schema[field].dataType, StringType) else field

def _array_source_typed(df, field, target_type):
    """Like _array_source, but for complex shapes (array<map<...>>,
    array<struct<...>>) instead of plain array<string>.

    HARDENING: if one side's pipeline serializes this field as a JSON string
    while the other already parses it to the real complex type -- a plausible
    schema-drift scenario between two independently-maintained bronze->silver
    jobs -- referencing struct/map fields directly on the string-typed side
    is a Spark AnalysisException (type mismatch) that kills the whole run,
    not a quiet wrong answer. Parsing defensively here means a drifted side
    degrades to "parse it" instead of "blow up the comparison job."
    """
    return f"from_json({field}, '{target_type}')" if isinstance(df.schema[field].dataType, StringType) else field

def _to_cat_set(df, field):
    src = _array_source(df, field)
    # trim() before the empty-string filter: a stray leading/trailing space
    # on a category token would otherwise survive as a distinct, non-empty
    # "value" and silently show up as a missing/extra pair instead of matching.
    return expr(rf"array_sort(array_distinct(filter(transform(coalesce({src}, cast(array() as array<string>)), cat -> trim(regexp_replace(regexp_extract(cat, '_(.*)$', 1), '\]$', ''))), c -> c is not null and c != '')))")

def _to_plain_set(df, field):
    src = _array_source(df, field)
    # cast every element to string: anchor/compare can carry the same field as
    # array<string> on one side and array<bigint> on the other (e.g. catalogIds),
    # and comparing across element types otherwise reads as a false MISMATCH.
    # trim() guards the same way as _to_cat_set above.
    return expr(rf"array_sort(array_distinct(transform(coalesce({src}, cast(array() as array<string>)), x -> trim(cast(x as string)))))")

# ── Key-based field comparison helpers ─────────────────────────────────────

def normalize_attributes_to_map(df, field_name):
    """
    Converts array<map<string,...>> into map<string, array<string>>, keyed by
    attribute key, collecting every value seen under a key into a sorted,
    deduped array (same collect-per-key pattern as normalize_keyed_struct_to_map).

    Uses map_entries(m) rather than map_keys(m)[0]/map_values(m)[0]: the
    latter keeps only the FIRST key of each array element and silently drops
    the rest the moment a single element ever carries more than one key/value
    pair. That's a quiet data-loss bug, not a loud one -- it would show up as
    a systematic, near-total MISMATCH (every row missing most of its keys on
    whichever side packs attributes that way), not a scattered one.
    map_entries always returns every pair a map holds, so this is safe
    whether each element carries one pair or many.

    HARDENING (added after manual spot-checks showed real production keys
    like "PRIMARY_CATEGORY_DSG " carrying a trailing space):
      - trim() on both key and value, applied once at the entry level before
        any grouping. Untrimmed, a key that picks up incidental whitespace
        on only one side of the comparison keys into a DIFFERENT map entry
        than its clean counterpart -- reading as one missing key on one side
        and one unrelated extra key on the other, even though every real
        value agrees. This is exactly the kind of false drift a byte-for-byte
        map comparison is vulnerable to.
      - defensive from_json parse (_array_source_typed) in case this column
        is ever serialized as a JSON string on one side instead of an
        already-parsed array<map<string,string>> -- see that helper's
        docstring for why this matters.
    """
    src = _array_source_typed(df, field_name, "array<map<string,string>>")
    entries = f"flatten(transform(coalesce({src}, array()), m -> map_entries(coalesce(m, map()))))"
    entries_trimmed = f"transform({entries}, e -> struct(trim(e.key) as key, trim(e.value) as value))"
    # Drop attribute entries whose key is in IGNORE_ATTRIBUTE_KEYS. Splitting
    # on "->" is not needed here because map_entries already gives us the key
    # and value as separate struct fields -- e.key is exactly the "first
    # part" of the "{key -> value}" representation.
    entries_filtered = f"filter({entries_trimmed}, e -> not array_contains({_IGNORE_KEYS_SQL_ARRAY}, e.key))"
    return expr(f"""
        map_from_entries(
          transform(
            array_distinct(transform({entries_filtered}, e -> e.key)),
            k -> struct(
              k as key,
              sort_array(array_distinct(
                transform(filter({entries_filtered}, e -> e.key = k), e -> e.value)
              )) as value
            )
          )
        )
    """)

def _keyed_struct_fields(df, field):
    """Names of the fields actually present in this DataFrame's array<struct> element type."""
    dt = df.schema[field].dataType
    if isinstance(dt, ArrayType) and isinstance(dt.elementType, StructType):
        return {f.name for f in dt.elementType.fields}
    return set()

def normalize_keyed_struct_to_map(df, field_name):
    """
    Converts array<struct{key, value|seq, storeId?, ...}> fields
    (customSkuAttributes, catgroupSeq) into map<string, array<string>>,
    keyed by "key" alone.

    Real-data issues drove this design, not just tidiness:
      - the struct shape isn't guaranteed to match between anchor and compare
        -- catgroupSeq in particular shows up as struct<key,seq> on one side
        and struct<key,seq,storeId,value> on the other. Referencing a field
        that doesn't exist on a given side is a compile-time error in Spark
        (FIELD_NOT_FOUND), so which fields are read is decided per-DataFrame
        from its actual schema, not assumed globally. Falls back to "seq"
        (cast to string) as the payload when "value" isn't present.
      - keying by "key||storeId" was tried first, but that misaligns the two
        sides' keyspaces whenever one side has no storeId concept at all
        (as above) -- every key would then read as MISMATCH purely because
        of the missing suffix, not real drift. Keying by "key" alone keeps
        both sides comparable regardless of whether storeId is present.
      - the SAME key can legitimately carry multiple distinct values (e.g.
        ALTERNATE_UPC repeats several times with different UPCs, potentially
        under different storeIds) alongside pure accidental duplicates
        (identical key+value repeated verbatim). Collecting a sorted,
        deduped array of every value seen under a key preserves the
        legitimate multi-value case while still collapsing exact dupes, and
        still catches genuine drift (e.g. key "2101": "Men's" vs "Youth").
    """
    if isinstance(df.schema[field_name].dataType, StringType):
        # Schema-drift fallback: this side never got parsed out of raw JSON.
        # We can't introspect a from_json-on-the-fly expression's field list
        # the way _keyed_struct_fields reads a real column's schema, so pull
        # the available field names out of the fallback schema string itself.
        fallback_schema = KEYED_STRUCT_JSON_SCHEMA.get(
            field_name, "array<struct<key:string,value:string>>"
        )
        src = f"from_json({field_name}, '{fallback_schema}')"
        available = set(re.findall(r"(\w+)\s*:", fallback_schema))
    else:
        src = field_name
        available = _keyed_struct_fields(df, field_name)

    if "value" in available:
        value_expr = "cast(x.value as string)"
    elif "seq" in available:
        value_expr = "cast(x.seq as string)"
    else:
        value_expr = "cast(x.key as string)"  # last resort, shouldn't happen

    # trim() both the grouping key and the payload value for the same reason
    # as normalize_attributes_to_map: incidental whitespace on only one side
    # (e.g. an upstream feed that pads "Y" -> " Y") must not read as a
    # missing/extra key or a mismatched value when the real content agrees.
    return expr(f"""
        map_from_entries(
          transform(
            array_distinct(transform(coalesce({src}, array()), x -> trim(x.key))),
            k -> struct(
              k as key,
              sort_array(array_distinct(
                transform(
                  filter(coalesce({src}, array()), x -> trim(x.key) = k),
                  x -> trim({value_expr})
                )
              )) as value
            )
          )
        )
    """)

def normalize_search_attributes_to_set(field_name):
    """
    Converts searchAttributes (space-separated string) into a sorted, lower-cased
    array of tokens. There's no delimiter distinguishing one attribute's value
    from the next (multi-word values like "Crew Neck" just run into neighboring
    tokens), so this can't be reconstructed into exact original values -- both
    anchor and compare go through the identical word-split, so a bag-of-words
    subset match still catches real drift without being fooled by that
    ambiguity. Lower-cased to avoid false MISMATCH from casing-only differences
    (e.g. "TRUE" vs "True" seen elsewhere in this same data).

    HARDENING: curly apostrophe (U+2019, "'") is normalized to a straight
    one before splitting. The two render identically but are different
    codepoints -- if one feed's export path normalizes quotes and the other
    doesn't, "Men's" (straight) vs "Men's" (curly) tokenize to two different
    words and read as bag-of-words drift over pure punctuation encoding, not
    real content.
    """
    # Embed the actual curly-quote character (not a '\u2019' escape sequence)
    # so this doesn't depend on how Spark SQL's string-literal parser handles
    # unicode escapes -- it's unambiguous either way.
    _curly_quote = "\u2019"
    quote_normalized = f"regexp_replace(coalesce({field_name}, ''), '{_curly_quote}', chr(39))"
    return expr(
        f"array_sort("
        f"  array_distinct("
        f"    filter("
        f"      transform("
        f"        split(lower({quote_normalized}), '\\\\s+'), "
        f"        v -> trim(v)"
        f"      ), "
        f"      v -> v != ''"
        f"    )"
        f"  )"
        f")"
    )

def normalize_image_field(field_name):
    """Strips any path/URL prefix and file extension, trims, lower-cases."""
    return expr(
        f"lower(trim(regexp_replace(regexp_replace(coalesce({field_name}, ''), '^.*/', ''), '\\\\.[a-zA-Z0-9]+$', '')))"
    )

def with_normalized_fields(df):
    df = df.withColumn("productSearchFlag_norm", coalesce(col("productSearchFlag"), lit(False)))

    for f in LONG_TEXT_FIELDS:
        df = df.withColumn(f"{f}_norm", regexp_replace(coalesce(col(f), lit("")), r"\s+", ""))
    for f in CAST_STRING_FIELDS:
        df = df.withColumn(f"{f}_norm", coalesce(col(f).cast("string"), lit("")))
    for f in IMAGE_FIELDS:
        df = df.withColumn(f"{f}_norm", normalize_image_field(f))
    for f in PLAIN_SET_FIELDS:
        df = df.withColumn(f"{f}_norm", _to_plain_set(df, f))
    for f in CAT_SET_FIELDS:
        df = df.withColumn(f"{f}_norm", _to_cat_set(df, f))

    # Key-based nested field comparisons
    # attributes: array<map> -> convert to map<string,string> for key-based comparison
    df = df.withColumn("attributes_norm", normalize_attributes_to_map(df, "attributes"))

    # customSkuAttributes / catgroupSeq: array<struct{key, value|seq, storeId?, ...}>
    # -> map<string, array<string>> keyed by "key" (see docstring)
    for f in KEYED_STRUCT_FIELDS:
        df = df.withColumn(f"{f}_norm", normalize_keyed_struct_to_map(df, f))

    # searchAttributes: space-separated string -> sorted array of lower-cased tokens
    if "searchAttributes" in df.columns:
        df = df.withColumn("searchAttributes_norm", normalize_search_attributes_to_set("searchAttributes"))

    # array<struct> (no maps) -> sort to make comparison order-insensitive
    for f in COMPLEX_ARRAY_STRUCT_FIELDS:
        df = df.withColumn(f"{f}_norm", expr(f"array_sort(coalesce({f}, array()))"))

    # plain structs (seo) need no norm column — compared directly, field order is
    # fixed by the schema so eqNullSafe on the struct itself is safe.
    return df

df_anchor_norm = with_normalized_fields(df_anchor)
df_compare_norm = with_normalized_fields(df_compare)

# Composite match key: mirrors attribute_comp.py's make_match_key so the same
# partnumber under two different parents on the two sides is treated as two
# unrelated records instead of force-paired by partnumber alone. NULL
# parentPartnumber (top-level records) is coalesced to a sentinel so both
# sides still align via a plain equi-join -- Spark's list-style join key
# treats NULL != NULL and would otherwise route those rows to extra/missing.
# The sentinel column is separate from the original parentPartnumber field
# (which stays NULL-preserving for per-row reporting) and is excluded from
# comparison_fields further down via EXCLUDE_COLUMNS.
_PARENT_NULL_SENTINEL = "__PP_NULL__"
df_anchor_norm = df_anchor_norm.withColumn(
    "_parent_match_key", coalesce(col("parentPartnumber"), lit(_PARENT_NULL_SENTINEL))
)
df_compare_norm = df_compare_norm.withColumn(
    "_parent_match_key", coalesce(col("parentPartnumber"), lit(_PARENT_NULL_SENTINEL))
)

# PERF: df_anchor_norm/df_compare_norm each add ~40 array/regex-derived
# columns and are referenced three times below (matched_df, extra_df,
# missing_df). Without persisting, each reference redoes all ~40
# normalizations from scratch on top of the already-persisted raw frames.
df_anchor_norm = df_anchor_norm.persist(CACHE_LEVEL)
df_compare_norm = df_compare_norm.persist(CACHE_LEVEL)
df_anchor_norm.count()  # materialize once, eagerly, instead of on first downstream use
df_compare_norm.count()

_JOIN_KEYS = ["partnumber", "_parent_match_key"]
matched_df = df_anchor_norm.alias("anchor").join(df_compare_norm.alias("compare"), on=_JOIN_KEYS, how="inner")
extra_df = df_anchor_norm.join(df_compare_norm.select(*_JOIN_KEYS), on=_JOIN_KEYS, how="left_anti").persist(CACHE_LEVEL)
missing_df = df_compare_norm.join(df_anchor_norm.select(*_JOIN_KEYS), on=_JOIN_KEYS, how="left_anti").persist(CACHE_LEVEL)

# ── ONE unified registry for every field needing non-default comparison ────
SET_FIELDS = PLAIN_SET_FIELDS + CAT_SET_FIELDS

NORMALIZED_FIELDS = {
    **{f: {"norm": f"{f}_norm", "match": "exact"} for f in LONG_TEXT_FIELDS},
    "productSearchFlag": {"norm": "productSearchFlag_norm", "match": "exact"},
    **{f: {"norm": f"{f}_norm", "match": "exact"} for f in CAST_STRING_FIELDS},
    **{f: {"norm": f"{f}_norm", "match": "exact"} for f in IMAGE_FIELDS},
    **{f: {"norm": f"{f}_norm", "match": "subset"} for f in SET_FIELDS},
    # Key-based nested field comparisons
    "attributes": {
        "norm": "attributes_norm",
        "match": "map_subset",
        "description": "Every anchor attribute key must exist in compare with an equal value; "
                        "compare carrying extra keys anchor doesn't have is not counted as drift"
    },
    **{f: {
        "norm": f"{f}_norm",
        "match": "map_subset",
        "description": "Every anchor key must exist in compare with an equal value (see "
                        "normalize_keyed_struct_to_map); compare carrying extra keys anchor "
                        "doesn't have is not counted as drift"
    } for f in KEYED_STRUCT_FIELDS},
    "searchAttributes": {
        "norm": "searchAttributes_norm",
        "match": "subset",
        "description": "Compares search keywords as a subset"
    },
    **{f: {"norm": f"{f}_norm", "match": "exact"} for f in COMPLEX_ARRAY_STRUCT_FIELDS},
}

# ── Item-level diff for the array/map fields flagged as "compared wrong" ───
# MATCH/MISMATCH above is a single pass/fail per field. It doesn't say WHICH
# items/keys actually differ, so a real gap (one attribute key genuinely
# missing) and a total mismatch look identical from the status column alone.
# DIFF_FIELDS gets a real breakdown instead: which items/keys are present on
# BOTH sides with the same value (matching), present in anchor only
# (missing), present in compare only (extra), and -- for keyed fields --
# present on both sides but with a different value (mismatched).
#
# This is deliberately NOT a positional "sort both arrays, then compare
# index i to index i" zip. That looks right on a clean side-by-side eyeball
# check (see MANUAL_COMPARISON_TEST.ipynb) because that sample happens to
# have almost the same key set on both sides -- but the moment one side is
# missing or has an extra item, every index after that point shifts out of
# alignment, and one real one-item gap reads as a wall of unrelated
# mismatches instead of the actual single missing/extra item. array_except/
# array_intersect below align items by their own value (or map key), not by
# position, so a shift on one side can never cascade into unrelated items.
DIFF_FIELDS = {
    "attributes": "map",
    "customSkuAttributes": "map",
    "catgroupSeq": "map",
    "searchAttributes": "array",
    "dsgCatgroups": "array",
    "ggCatgroups": "array",
    "catalogIds": "array",
}

# DIFF logic: expressed as native Spark SQL higher-order functions
# (array_intersect / array_except / filter / exists) so the whole expression
# tree is codegen-friendly instead of paying per-row Python UDF overhead on
# millions of rows. Same semantics as the previous Python version:
#   - arrays: align by value, not by position (a shift on one side never
#     cascades into unrelated items)
#   - maps: align by map key, and a key is "matching" when at least one value
#     anchor holds for it overlaps with compare's values for that same key.
def build_diff_exprs(norm_field, shape):
    """Returns (matching, missing, mismatched, extra) as sorted STRING columns.
    Convention (per the summary-table semantics):
      - missing = values on the COMPARE side but not in ANCHOR
      - extra   = values on the ANCHOR side but not in COMPARE
    """
    a = f"anchor.{norm_field}"
    c = f"compare.{norm_field}"

    if shape == "array":
        matching = f"array_sort(array_intersect({a}, {c}))"
        missing = f"array_sort(array_except({c}, {a}))"      # in compare, not in anchor
        mismatched = "array()"  # N/A: plain arrays have no "same slot, different value" concept
        extra = f"array_sort(array_except({a}, {c}))"        # in anchor, not in compare
    else:  # map<string, array<string>>, keyed by map_keys()
        a_keys, c_keys = f"map_keys({a})", f"map_keys({c})"
        common_keys = f"filter({a_keys}, k -> array_contains({c_keys}, k))"
        # any-overlap-on-value-list => matching; empty overlap => mismatched
        matching = f"array_sort(filter({common_keys}, k -> size(array_intersect({a}[k], {c}[k])) > 0))"
        mismatched = f"array_sort(filter({common_keys}, k -> size(array_intersect({a}[k], {c}[k])) = 0))"
        missing = f"array_sort(array_except({c_keys}, {a_keys}))"  # keys in compare, not in anchor
        extra = f"array_sort(array_except({a_keys}, {c_keys}))"    # keys in anchor, not in compare

    return tuple(expr(f"cast({e} as string)") for e in (matching, missing, mismatched, extra))

# ── Per-row similarity (Jaccard-based, 0-100) ──────────────────────────────
# Applied to EVERY comparison field, not just the complex ones. For a scalar
# it collapses to 100/0 (so its per-field avg still equals the old match_pct);
# for arrays and maps it gives partial credit via Jaccard so a "mostly the
# same" row scores much higher than a "completely disjoint" row instead of
# both falling into the same MISMATCH bucket.
#   - scalar (int/bool/string/struct): 100 if <=> else 0.
#   - array<scalar>: |A ∩ B| / |A ∪ B| * 100.
#   - array<struct|map>: same, but each element cast to string first so
#     array_intersect/array_union can compare them.
#   - map<string, array<string>>: flatten each key's value list into
#     "key||value" pairs, then Jaccard over pairs (drift in value shows up
#     as fully disjoint pairs; extra values under a shared key reduce
#     similarity smoothly).
#   - text: word-set Jaccard on the RAW column (lowercased, whitespace-split).
#     Not the _norm column -- LONG_TEXT_FIELDS strip whitespace to a single
#     token, which would make word-Jaccard meaningless there.
# Null-safety: both null => 100. One null, one not => 0.
SIMILARITY_FIELDS = {
    **{f: "array" for f in CAT_SET_FIELDS},
    "customSkuAttributes": "map",
    "catgroupSeq": "map",
    "attributes": "map",
}

def _similarity_core(a_ref, c_ref, shape):
    """Returns the SQL CASE expression (no cast/round) for shape-specific similarity."""
    if shape == "scalar":
        # Explicit null handling instead of a bare `<=>`. `<=>` is documented
        # null-safe, but on struct-typed columns whose inner fields are null
        # (real production case: `seo` with any missing sub-field) Spark can
        # propagate a NULL out of the nested comparison, and the outer CASE
        # WHEN NULL falls through as NULL rather than 0.0/100.0. That was
        # showing up as NULL similarity_pct on rows that should have scored
        # 100 (both sides null) or 0 (drift). Split the null cases out first
        # so both-null explicitly maps to 100.0 before we ever try to compare
        # values, then fall back to `<=>` for the value-vs-value path.
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN {a_ref} <=> {c_ref} THEN 100.0
              ELSE 0.0
            END
        """

    if shape == "array":
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN size(array_union({a_ref}, {c_ref})) = 0 THEN 100.0
              ELSE size(array_intersect({a_ref}, {c_ref})) * 100.0 / size(array_union({a_ref}, {c_ref}))
            END
        """

    if shape == "array_struct":
        a_strs = f"transform(coalesce({a_ref}, array()), s -> cast(s as string))"
        c_strs = f"transform(coalesce({c_ref}, array()), s -> cast(s as string))"
        return f"""
            CASE
              WHEN size(array_union({a_strs}, {c_strs})) = 0 THEN 100.0
              ELSE size(array_intersect({a_strs}, {c_strs})) * 100.0 / size(array_union({a_strs}, {c_strs}))
            END
        """

    if shape == "map":
        a_pairs = f"flatten(transform(map_entries({a_ref}), e -> transform(e.value, v -> concat_ws('||', e.key, v))))"
        c_pairs = f"flatten(transform(map_entries({c_ref}), e -> transform(e.value, v -> concat_ws('||', e.key, v))))"
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN size(array_union({a_pairs}, {c_pairs})) = 0 THEN 100.0
              ELSE size(array_intersect({a_pairs}, {c_pairs})) * 100.0 / size(array_union({a_pairs}, {c_pairs}))
            END
        """

    if shape == "text":
        a_words = f"filter(array_distinct(split(lower(coalesce({a_ref}, '')), '\\\\s+')), w -> w != '')"
        c_words = f"filter(array_distinct(split(lower(coalesce({c_ref}, '')), '\\\\s+')), w -> w != '')"
        return f"""
            CASE
              WHEN size(array_union({a_words}, {c_words})) = 0 THEN 100.0
              ELSE size(array_intersect({a_words}, {c_words})) * 100.0 / size(array_union({a_words}, {c_words}))
            END
        """

    raise ValueError(f"Unknown similarity shape: {shape}")

def build_similarity_expr(a_ref, c_ref, shape):
    """Returns a DOUBLE column (0.0 - 100.0) rounded to 2dp."""
    return expr(f"cast(round({_similarity_core(a_ref, c_ref, shape)}, 2) as double)")

# ── Per-row value-level counts (MATCH / MISMATCH / MISSING / EXTRA / TOTAL) ─
# For each row and each field, classify the values into buckets so the
# summary table can sum(bucket) across all rows and give a value-level count
# per field instead of a row-level pass/fail count.
#   - scalar: one "slot" per row -> exactly one of match/mismatch/missing/
#     extra is 1; total is 1 (0 if both null).
#   - array<scalar>: count matching items (intersect), missing items
#     (compare - anchor), extra items (anchor - compare); mismatch = 0
#     (no positional "same slot, different value" concept for flat arrays).
#     total = union size.
#   - array<struct|map>: same as array<scalar> but each element cast to
#     string first so array_intersect/array_union can compare them.
#   - text: word-set counts on the raw column (same accounting as arrays).
#   - map<string, array<string>>: match = keys in both with overlapping
#     values; mismatch = keys in both with disjoint values; missing = keys
#     in compare not in anchor; extra = keys in anchor not in compare;
#     total = union of all touched keys.
# Invariant: m + mm + miss + ext == total (both sides), always.
def _valuestat_core(a_ref, c_ref, shape):
    """Returns (match, mismatch, missing, extra, total) as SQL int expressions."""
    if shape == "scalar":
        # EXTRA doesn't apply to a scalar column: a scalar holds one value
        # per row, so "anchor has a value compare doesn't" is a compare-side
        # gap, not an anchor-side surplus. Those rows are dropped from the
        # field-level accounting entirely -- ext is a constant 0 and total
        # only counts rows where compare has a value (both-non-null MATCH/
        # MISMATCH, or anchor-null MISSING). similarity_pct then reduces to
        # MATCH / (MATCH + MISMATCH + MISSING), which is what we want for
        # day-over-day tracking on scalars: MATCH growing pulls from the
        # remaining two directions the compare side can actually be off in.
        # Array/map/text shapes below keep EXTRA because they can hold
        # multiple values per row and an anchor-only item there is real
        # extra content, not just a compare-side null.
        m = f"CASE WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0 WHEN {a_ref} = {c_ref} THEN 1 ELSE 0 END"
        mm = f"CASE WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0 WHEN {a_ref} <> {c_ref} THEN 1 ELSE 0 END"
        miss = f"CASE WHEN {a_ref} IS NULL AND {c_ref} IS NOT NULL THEN 1 ELSE 0 END"
        ext = "0"
        total = f"CASE WHEN {c_ref} IS NOT NULL THEN 1 ELSE 0 END"
        return m, mm, miss, ext, total

    if shape in ("array", "array_struct", "text"):
        if shape == "array_struct":
            a_arr = f"transform(coalesce({a_ref}, array()), s -> cast(s as string))"
            c_arr = f"transform(coalesce({c_ref}, array()), s -> cast(s as string))"
        elif shape == "text":
            a_arr = f"filter(array_distinct(split(lower(coalesce({a_ref}, '')), '\\\\s+')), w -> w != '')"
            c_arr = f"filter(array_distinct(split(lower(coalesce({c_ref}, '')), '\\\\s+')), w -> w != '')"
        else:
            a_arr = f"coalesce({a_ref}, array())"
            c_arr = f"coalesce({c_ref}, array())"
        m = f"size(array_intersect({a_arr}, {c_arr}))"
        mm = "0"
        miss = f"size(array_except({c_arr}, {a_arr}))"   # compare - anchor
        ext = f"size(array_except({a_arr}, {c_arr}))"    # anchor - compare
        total = f"size(array_union({a_arr}, {c_arr}))"
        return m, mm, miss, ext, total

    if shape == "map":
        a_keys = f"coalesce(map_keys({a_ref}), array())"
        c_keys = f"coalesce(map_keys({c_ref}), array())"
        common_keys = f"filter({a_keys}, k -> array_contains({c_keys}, k))"
        m = f"size(filter({common_keys}, k -> size(array_intersect({a_ref}[k], {c_ref}[k])) > 0))"
        mm = f"size(filter({common_keys}, k -> size(array_intersect({a_ref}[k], {c_ref}[k])) = 0))"
        miss = f"size(array_except({c_keys}, {a_keys}))"   # keys in compare - anchor
        ext = f"size(array_except({a_keys}, {c_keys}))"    # keys in anchor - compare
        total = f"size(array_union({a_keys}, {c_keys}))"
        return m, mm, miss, ext, total

    raise ValueError(f"Unknown valuestat shape: {shape}")

EXCLUDE_COLUMNS = {"partnumber", "load_timestamp", "_parent_match_key"} | {v["norm"] for v in NORMALIZED_FIELDS.values()}
SUBSET_EMPTY_AS_MATCH = False  # empty anchor set = MISMATCH (a real gap)

def is_simple_or_simple_array(field):
    """Allow scalars and arrays of scalars; reject arrays of structs, maps, structs
    (COMPLEX_STRUCT_FIELDS / NORMALIZED_FIELDS are handled explicitly elsewhere)."""
    dt = field.dataType
    if isinstance(dt, (MapType, StructType)):
        return False
    if isinstance(dt, ArrayType):
        return not isinstance(dt.elementType, (StructType, MapType, ArrayType))
    return True

schema_fields = {f.name: f for f in df_anchor_norm.schema.fields}

skipped_complex = [
    name for name, f in schema_fields.items()
    if name not in EXCLUDE_COLUMNS
    and name not in NORMALIZED_FIELDS
    and name not in COMPLEX_STRUCT_FIELDS
    and not is_simple_or_simple_array(f)
]

# comparison_fields picks up every remaining simple column AND the plain-struct
# fields (seo), since structs with no maps compare fine with eqNullSafe directly
comparison_fields = [
    c for c in df_anchor_norm.columns
    if c not in EXCLUDE_COLUMNS and c not in NORMALIZED_FIELDS and c not in skipped_complex
]

# Safety net: catch anything unaccounted for (schema drift, new columns, etc.)
all_columns = set(df_anchor_norm.columns)
accounted_for = set(comparison_fields) | set(NORMALIZED_FIELDS) | set(skipped_complex) | EXCLUDE_COLUMNS
unaccounted = sorted(all_columns - accounted_for)
if unaccounted:
    print(f"⚠️  {len(unaccounted)} column(s) were unaccounted for — adding to comparison_fields (direct eqNullSafe):")
    for name in unaccounted:
        print(f"   + {name} ({schema_fields[name].dataType.simpleString()})")
    comparison_fields += unaccounted

all_fields = comparison_fields + list(NORMALIZED_FIELDS.keys())
status_cols = [f"{f}_status" for f in all_fields]

print(f"\nComparing {len(all_fields)} fields "
      f"({len(comparison_fields)} direct + {len(NORMALIZED_FIELDS)} normalized)")
if skipped_complex:
    print(f"\n⚠️  Still skipped {len(skipped_complex)} columns:")
    for name in skipped_complex:
        print(f"   - {name} ({schema_fields[name].dataType.simpleString()})")
else:
    print("No fields skipped.")

empty_result = "MATCH" if SUBSET_EMPTY_AS_MATCH else "MISMATCH"

# ── Comparison-decision expressions (SQL, not UDFs) ─────────────────────────
# Native higher-order functions (exists / array_intersect / <=>) so the
# whole comparison stays inside Spark's codegen instead of paying per-row
# Python overhead. Semantics: any-overlap wins.
#   - subset (plain arrays): MATCH if at least one anchor item exists in
#     compare (any overlap between the two flat arrays).
#   - map_subset (map<string, array<string>>): MATCH if at least one anchor
#     key exists in compare with at least one overlapping value under that
#     key. Anchor holding extra values compare doesn't have is not drift,
#     compare holding extra values is not drift.
#   - map_exact: two maps identical (null-safe equality).
def build_comparison_expr(cfg):
    """Builds the comparison column for a field based on its match type."""
    norm_field = cfg["norm"]
    match_type = cfg.get("match", "exact")
    a = f"anchor.{norm_field}"
    c = f"compare.{norm_field}"

    if match_type == "exact":
        return when(col(a).eqNullSafe(col(c)), lit("MATCH")).otherwise(lit("MISMATCH"))

    elif match_type == "subset":
        return expr(f"""
            CASE
              WHEN size({a}) = 0 THEN '{empty_result}'
              WHEN size(array_intersect({a}, {c})) > 0 THEN 'MATCH'
              ELSE 'MISMATCH'
            END
        """)

    elif match_type == "map_exact":
        return expr(f"CASE WHEN {a} <=> {c} THEN 'MATCH' ELSE 'MISMATCH' END")

    elif match_type == "map_subset":
        return expr(f"""
            CASE
              WHEN size({a}) = 0 THEN '{empty_result}'
              WHEN exists(
                map_keys({a}),
                k -> array_contains(map_keys({c}), k)
                     AND size(array_intersect({a}[k], {c}[k])) > 0
              ) THEN 'MATCH'
              ELSE 'MISMATCH'
            END
        """)

    else:
        raise ValueError(f"Unknown match type: {match_type}")

direct_exprs = [
    when(col(f"anchor.{field}").eqNullSafe(col(f"compare.{field}")), lit("MATCH")).otherwise(lit("MISMATCH")).alias(f"{field}_status")
    for field in comparison_fields
]

normalized_exprs = [
    build_comparison_expr(cfg).alias(f"{field}_status")
    for field, cfg in NORMALIZED_FIELDS.items()
]

diff_exprs = []
for field, shape in DIFF_FIELDS.items():
    matching_e, missing_e, mismatched_e, extra_e = build_diff_exprs(NORMALIZED_FIELDS[field]["norm"], shape)
    diff_exprs += [
        matching_e.alias(f"{field}_matching"),
        missing_e.alias(f"{field}_missing"),
        mismatched_e.alias(f"{field}_mismatched"),
        extra_e.alias(f"{field}_extra"),
    ]

# Per-row similarity for EVERY field. Pick the right shape based on field
# category so simple scalars get 100/0 (avg == old match_pct) and complex
# fields get Jaccard partial credit.
def _similarity_shape_and_refs(field, cfg=None):
    """Returns (a_ref, c_ref, shape) for a given field.
    Uses the raw column for LONG_TEXT_FIELDS (word-set Jaccard needs
    whitespace that the _norm column strips out); otherwise uses the norm
    column when there is one, and the raw column for direct comparison_fields.
    """
    if cfg is not None:
        # normalized field
        if field in LONG_TEXT_FIELDS:
            # raw text for word-set Jaccard -- _norm has whitespace stripped
            return (f"anchor.{field}", f"compare.{field}", "text")
        norm = cfg["norm"]
        if field in SIMILARITY_FIELDS:
            shape = SIMILARITY_FIELDS[field]
        elif field in COMPLEX_ARRAY_STRUCT_FIELDS:
            shape = "array_struct"
        elif field == "searchAttributes" or field in PLAIN_SET_FIELDS:
            shape = "array"
        else:
            # productSearchFlag, catentryId, fullImage, etc.
            shape = "scalar"
        return (f"anchor.{norm}", f"compare.{norm}", shape)

    # direct comparison_field -- infer shape from the DataFrame schema
    dt = schema_fields[field].dataType
    if isinstance(dt, ArrayType):
        if isinstance(dt.elementType, (StructType, MapType, ArrayType)):
            shape = "array_struct"
        else:
            shape = "array"
    else:
        shape = "scalar"
    return (f"anchor.{field}", f"compare.{field}", shape)

similarity_exprs = []
for field in comparison_fields:
    a_ref, c_ref, shape = _similarity_shape_and_refs(field)
    similarity_exprs.append(build_similarity_expr(a_ref, c_ref, shape).alias(f"{field}_similarity"))
for field, cfg in NORMALIZED_FIELDS.items():
    a_ref, c_ref, shape = _similarity_shape_and_refs(field, cfg)
    similarity_exprs.append(build_similarity_expr(a_ref, c_ref, shape).alias(f"{field}_similarity"))

status_exprs = direct_exprs + normalized_exprs

matched_status = (
    matched_df
    .select("*", *status_exprs, *diff_exprs, *similarity_exprs)
    .withColumn(
        "row_status",
        when(array_contains(array(*[col(c) for c in status_cols]), "MISMATCH"), lit("MISMATCH")).otherwise(lit("MATCH")),
    )
    # ── Document-availability gate ─────────────────────────────────────────
    # A row where either side's `type` is NULL is treated as an "unavailable
    # document": we have a partnumber match but at least one side never
    # populated the document type, so any field-level comparison on that row
    # is comparing against a hole and would skew both the per-field and
    # dataset-wide similarity numbers. Flag it here (visible per-row for
    # audit) and filter it out of every aggregation below so it does not
    # contribute to the report.
    .withColumn(
        "document_available",
        expr("anchor.type IS NOT NULL AND compare.type IS NOT NULL"),
    )
)

# PERF: matched_status is the widest, most expensive frame in the notebook
# (275 status columns computed row-by-row over the joined anchor/compare
# data). It is referenced by: the MATCH/MISMATCH groupBy below, the detail
# write (which unpivots all 275 fields via stack()), and the per-field
# agg() used for the summary table. Previously each of those recomputed the
# full join + all 275 `when()`/`forall` expressions independently. Persist
# once, right after it's built, and everything downstream reuses it.
matched_status = matched_status.persist(CACHE_LEVEL)

# COMMAND ----------
# Databricks position: 8.25
#matched_status.display()

# COMMAND ----------
# Databricks position: 8.5
extra_status = extra_df.select("partnumber").withColumn("row_status", lit("EXTRA"))
missing_status = missing_df.select("partnumber").withColumn("row_status", lit("MISSING"))

# PERF + BUG FIX: this single groupBy (over the now-persisted matched_status)
# replaces four separate full-lineage .count() calls that used to exist here:
#   match_count = matched_df.count()                                    # recomputed the whole join
#   mismatch_count = matched_status.select("partnumber")
#                     .withColumn("row_status", lit("MISMATCH")).count()  # BUG: this just
#                     # relabels every row "MISMATCH" and counts them all --
#                     # it silently returned the *total* row count, not the
#                     # actual mismatch count, while also triggering a full
#                     # recompute of matched_status.
# match/mismatch now come straight out of the one groupBy below.
# Unavailable documents (document_available == false) are excluded here so
# missing-type rows are not counted as either MATCH or MISMATCH in the report.
row_stats = {
    r["row_status"]: r["count"]
    for r in matched_status.filter(col("document_available"))
                           .groupBy("row_status").count().collect()
}

match_count = row_stats.get("MATCH", 0)
mismatch_count = row_stats.get("MISMATCH", 0)
unavailable_count = matched_status.filter(~col("document_available")).count()

missing_count = missing_df.count()
extra_count = extra_df.count()

print(f"Row-level statistics (RunID: {run_id}):")
print(f"  Match:       {match_count:,}")
print(f"  Mismatch:    {mismatch_count:,}")
print(f"  Missing:     {missing_count:,}")
print(f"  Extra:       {extra_count:,}")
print(f"  Unavailable: {unavailable_count:,}   (excluded from field-level report)")

# ── Dataset-wide value-level breakdown ─────────────────────────────────────
# The row-level counts above only answer "did the row match overall?" -- a
# row with 269 out of 270 fields matching still counts the same as a row
# where every field disagrees. This block sums the per-field valuestat
# expressions across EVERY field and every row, so we can report what
# fraction of the *values* landed in each bucket. Ratios always sum to 100%
# by construction (m + mm + miss + ext == total in every valuestat_core
# branch). Single pass over the already-persisted matched_status.
_match_terms, _mismatch_terms, _missing_terms, _extra_terms, _total_terms = [], [], [], [], []
for _field in comparison_fields:
    _a, _c, _shape = _similarity_shape_and_refs(_field)
    _m, _mm, _miss, _ext, _tot = _valuestat_core(_a, _c, _shape)
    _match_terms.append(_m); _mismatch_terms.append(_mm)
    _missing_terms.append(_miss); _extra_terms.append(_ext); _total_terms.append(_tot)
for _field, _cfg in NORMALIZED_FIELDS.items():
    _a, _c, _shape = _similarity_shape_and_refs(_field, _cfg)
    _m, _mm, _miss, _ext, _tot = _valuestat_core(_a, _c, _shape)
    _match_terms.append(_m); _mismatch_terms.append(_mm)
    _missing_terms.append(_miss); _extra_terms.append(_ext); _total_terms.append(_tot)

_overall = matched_status.filter(col("document_available")).agg(
    expr(f"sum({' + '.join(_match_terms)})").alias("m"),
    expr(f"sum({' + '.join(_mismatch_terms)})").alias("mm"),
    expr(f"sum({' + '.join(_missing_terms)})").alias("miss"),
    expr(f"sum({' + '.join(_extra_terms)})").alias("ext"),
    expr(f"sum({' + '.join(_total_terms)})").alias("tot"),
).first().asDict()

overall_match_values    = _overall.get("m")    or 0
overall_mismatch_values = _overall.get("mm")   or 0
overall_missing_values  = _overall.get("miss") or 0
overall_extra_values    = _overall.get("ext")  or 0
overall_total_values    = _overall.get("tot")  or 0

def _pct(v):
    return (100.0 * v / overall_total_values) if overall_total_values else 0.0

overall_match_pct    = _pct(overall_match_values)
overall_mismatch_pct = _pct(overall_mismatch_values)
overall_missing_pct  = _pct(overall_missing_values)
overall_extra_pct    = _pct(overall_extra_values)

print(f"\nDataset-wide value-level similarity (RunID: {run_id}, {overall_total_values:,} values compared):")
print(f"  Match:    {overall_match_pct:6.2f}%   ({overall_match_values:,})")
print(f"  Mismatch: {overall_mismatch_pct:6.2f}%   ({overall_mismatch_values:,})")
print(f"  Missing:  {overall_missing_pct:6.2f}%   ({overall_missing_values:,})")
print(f"  Extra:    {overall_extra_pct:6.2f}%   ({overall_extra_values:,})")


# COMMAND ----------
# Databricks position: 9.0
# Only persist drift: per-field mismatches (long format) plus row-level missing/extra
# partnumbers. Fully matching rows are intentionally omitted - their absence is the signal.
# Backtick-quote column names to handle special chars (e.g. parentCatgroup0-9)
stack_args = ", ".join(
    f"'{field}', `{field}_status`, CAST(`anchor`.`{field}` AS STRING), CAST(`compare`.`{field}` AS STRING), "
    + (
        f"`{field}_matching`, `{field}_missing`, `{field}_mismatched`, `{field}_extra`, "
        if field in DIFF_FIELDS else
        "CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING), "
    )
    + f"`{field}_similarity`"
    for field in all_fields
)
stack_expr = (
    f"stack({len(all_fields)}, {stack_args}) as "
    f"(field, status, anchor_value, compare_value, matching_value, missing_value, mismatched_value, extra_value, similarity_pct)"
)

mismatch_fields_df = (
    matched_status
    .filter(col("document_available"))         # exclude unavailable documents from drift detail
    .filter(col("row_status") == "MISMATCH")
    .select("partnumber", expr(stack_expr))
    .filter(col("status") == "MISMATCH")
    .select(
        "partnumber",
        lit("MISMATCH").alias("row_status"),
        "field",
        "anchor_value",
        "compare_value",
        "matching_value",
        "missing_value",
        "mismatched_value",
        "extra_value",
        "similarity_pct",
    )
)

row_level_df = (
    extra_status.select("partnumber", "row_status")
    .unionByName(missing_status.select("partnumber", "row_status"))
    .withColumn("field", lit(None).cast("string"))
    .withColumn("anchor_value", lit(None).cast("string"))
    .withColumn("compare_value", lit(None).cast("string"))
    .withColumn("matching_value", lit(None).cast("string"))
    .withColumn("missing_value", lit(None).cast("string"))
    .withColumn("mismatched_value", lit(None).cast("string"))
    .withColumn("extra_value", lit(None).cast("string"))
    .withColumn("similarity_pct", lit(None).cast("double"))
)

detail_result_df = (
    mismatch_fields_df
    .unionByName(row_level_df)
    .withColumn("run_id", lit(run_id))
    .withColumn("run_timestamp", lit(run_timestamp))
    .withColumn("run_date", lit(run_date))
)

detail_row_count = detail_result_df.count()

(
    detail_result_df.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"run_date = '{run_date}'")
    .option("mergeSchema", "true")
    .partitionBy("run_date")
    .saveAsTable(detail_table)
)

print(f"Wrote {detail_row_count:,} drift rows (mismatched fields + missing/extra partnumbers) to {detail_table}")

# COMMAND ----------
# Databricks position: 10.0
# Field grouping flag, mirrors the V2 report categories
def _field_category(name):
    n = name.lower()
    if n.endswith("webactive"):
        return "Web Active Flag"
    if "overrides" in n:
        return "Overrides"
    if n.startswith("kafkapricelist") or "priceindicators" in n:
        return "Pricing"
    if n.startswith("ranking_"):
        return "Ranking"
    if n.startswith("salesdata") or n.endswith("quantitysold") or n.endswith("totalpricesold"):
        return "Sales"
    if n.endswith("facets") or n.endswith("attributes"):
        return "Attribution"
    if ("catgroup" in n or "leafcategories" in n or "seourl" in n
            or n.startswith("primarycategories") or n in {"productgroup", "productsearchflag", "seo"}):
        return "Caddyshack"
    return "MDM"

def _agg_exprs_for_field(field, a_ref, c_ref, shape):
    """Value-level MATCH/MISMATCH/MISSING/EXTRA/TOTAL summed across rows.

    similarity_pct is deliberately NOT aggregated here. The previous version
    reported AVG(per-row similarity), which inflated the field score in two
    ways and made similarity_pct != MATCH/total × 100:
      - Sparse scalars (e.g. onOrder, ggPublishOverride): rows where BOTH
        sides are null score per-row similarity = 100 (both-null is treated
        as "nothing to disagree on") but are excluded from the value-level
        counts (total=0 for that row). Averaging in those 100s dragged the
        field number toward 100 for any field where most rows are both-null.
        For onOrder that meant reporting 70% when only 31% of the actual
        values agreed.
      - Array/map fields (e.g. leafCategories): AVG(per-row Jaccard) is not
        equal to SUM(|A ∩ B|) / SUM(|A ∪ B|) in general. Rows with small
        universes get the same weight as rows with large ones, so a field
        with many nearly-empty rows drifts higher than the value-level
        count actually is.
    Both effects go away when we compute similarity_pct = MATCH / total × 100
    from the summed counts downstream instead of averaging per-row scores."""
    m, mm, miss, ext, total = _valuestat_core(a_ref, c_ref, shape)
    return [
        expr(f"sum({m})").alias(f"{field}__MATCH"),
        expr(f"sum({mm})").alias(f"{field}__MISMATCH"),
        expr(f"sum({miss})").alias(f"{field}__MISSING"),
        expr(f"sum({ext})").alias(f"{field}__EXTRA"),
        expr(f"sum({total})").alias(f"{field}__TOTAL"),
    ]

agg_exprs = []
for field in comparison_fields:
    a_ref, c_ref, shape = _similarity_shape_and_refs(field)
    agg_exprs += _agg_exprs_for_field(field, a_ref, c_ref, shape)
for field, cfg in NORMALIZED_FIELDS.items():
    a_ref, c_ref, shape = _similarity_shape_and_refs(field, cfg)
    agg_exprs += _agg_exprs_for_field(field, a_ref, c_ref, shape)

field_counts = matched_status.filter(col("document_available")).agg(*agg_exprs).first().asDict()

def _similarity_pct(match_count, total_count):
    """Field-level similarity: fraction of comparable VALUES that agreed.
    A field where no values ever showed up on either side (total = 0) reads
    as 100.0 by convention -- there's nothing to disagree on, matching the
    empty-set case elsewhere in this notebook. Rounded to 2 dp so it matches
    the previous column's presentation and stays comparable across runs."""
    if not total_count:
        return 100.0
    return round(100.0 * match_count / total_count, 2)

summary_rows = []
for field in all_fields:
    match_v = field_counts.get(f"{field}__MATCH") or 0
    total_v = field_counts.get(f"{field}__TOTAL") or 0
    # SUM(...) over zero rows returns NULL, not 0 -- coalesce here so
    # downstream math and schema inference never see a None.
    summary_rows.append({
        "field_category": _field_category(field),
        "field": field,
        "MATCH": match_v,
        "MISMATCH": field_counts.get(f"{field}__MISMATCH") or 0,
        "MISSING": field_counts.get(f"{field}__MISSING") or 0,
        "EXTRA": field_counts.get(f"{field}__EXTRA") or 0,
        "total": total_v,
        # similarity_pct = MATCH * 100 / total, computed from the SUMMED
        # counts (see _agg_exprs_for_field for why we no longer aggregate
        # AVG(per-row similarity) instead).
        "similarity_pct": _similarity_pct(match_v, total_v),
        "run_id": run_id,
        "run_timestamp": run_timestamp,
        "run_date": run_date,
    })

# Explicit schema instead of relying on inference: if any count were NULL for
# every row (belt-and-suspenders with the coalesce above), spark.createDataFrame's
# type inference can't determine a type at all and throws CANNOT_DETERMINE_TYPE.
summary_schema = StructType([
    StructField("field_category", StringType()),
    StructField("field", StringType()),
    StructField("MATCH", LongType()),
    StructField("MISMATCH", LongType()),
    StructField("MISSING", LongType()),
    StructField("EXTRA", LongType()),
    StructField("total", LongType()),
    StructField("similarity_pct", DoubleType(), nullable=True),
    StructField("run_id", StringType()),
    StructField("run_timestamp", TimestampType()),
    StructField("run_date", DateType()),
])

summary_result = (
    spark.createDataFrame(summary_rows, schema=summary_schema)
    .orderBy("field_category","field")
)

print(f"Preview before writing field-level summary to {summary_table}:")
display(summary_result.select("field_category", "field", "MATCH", "MISMATCH", "MISSING", "EXTRA", "total", "similarity_pct").orderBy("field_category", "field"))

(
    summary_result.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"run_date = '{run_date}'")
    .option("mergeSchema", "true")
    .partitionBy("run_date")
    .saveAsTable(summary_table)
)

print(f"Wrote field-level summary to {summary_table}")

# COMMAND ----------
# Databricks position: 11.0
execution_time_seconds = time.perf_counter() - start_time
compare_table_schema = [f"{f.name}:{f.dataType.simpleString()}" for f in df_compare.schema.fields]

metadata_row = [{
    "run_id": run_id,
    "run_timestamp": run_timestamp,
    "run_date": run_date,
    "anchor_table": f"{catalog_name}.{silver_schema_name}.{anchor_table_name}",
    "compare_table": f"{catalog_name}.{silver_schema_name}.{compare_table_name}",
    "compare_table_schema": compare_table_schema,
    "anchor_record_count": anchor_count,
    "compare_record_count": compare_count,
    "match_count": match_count,
    "mismatch_count": mismatch_count,
    "missing_count": missing_count,
    "extra_count": extra_count,
    "unavailable_count": unavailable_count,
    "execution_time_seconds": execution_time_seconds,
    "user": run_user,
    "notebook_path": notebook_path,
}]

(
    spark.createDataFrame(metadata_row)
    .write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"run_date = '{run_date}'")
    .option("mergeSchema", "true")
    .partitionBy("run_date")
    .saveAsTable(metadata_table)
)

print(f"Run {run_id} complete in {execution_time_seconds:.2f}s")


# COMMAND ----------
# Databricks position: 12.0
#display(
#    summary_result.select("field", "MATCH", "MISMATCH", "total", "similarity_pct", "MISSING", "EXTRA").orderBy("field")
#)

# ===== position 13.0 (new) =====
# PERF: release cached storage explicitly. On serverless this also matters
# because leftover persisted frames can otherwise linger and eat into the
# next run's budget if the session is reused.
for _df in (df_anchor, df_compare, df_anchor_norm, df_compare_norm, matched_status, extra_df, missing_df):
    _df.unpersist()S

Starting comparison run: 20260907_215400_996be16f
User: leonardo.paschoal@dcsg.com | Notebook: /Users/leonardo.paschoal@dcsg.com/sdds_index_comparison/catalog-compare-run-id-sku-optimized
Anchor (catalog-stream-dbx-silver) records:  1,812,825 (extraction_date=2026-09-07)
Compare (catalog-load-dbx-silver) records: 727,662 (extraction_date=2026-09-07)

Comparing 275 fields (246 direct + 29 normalized)

⚠️  Still skipped 1 columns:
   - productGroup (array<struct<id:string,seq:bigint,sequence:bigint>>)
Row-level statistics (RunID: 20260907_215400_996be16f):
  Match:    0
  Mismatch: 727,662
  Missing:  0
  Extra:    1,085,163

Dataset-wide value-level similarity (RunID: 20260907_215400_996be16f, 399,746,872 values compared):
  Match:     71.67%   (286,499,534)
  Mismatch:   3.69%   (14,731,228)
  Missing:    7.82%   (31,275,489)
  Extra:     16.82%   (67,240,621)
Wrote 30,432,440 drift rows (mismatched fields + missing/extra partnumbers) to `dev_sdsc_db`.`sdds_gold`.`field_level_compariso

field_category,field,MATCH,MISMATCH,MISSING,EXTRA,total,similarity_pct
Attribution,attributes,20856488,313991,409063,4075045,25654587,81.17
Attribution,customSkuAttributes,24698416,903992,335364,6095631,32033403,75.99
Attribution,defAttributes,1240629,0,19422,2286,1262337,98.42
Attribution,floatFacets,3153724,0,137379,141477,3432580,93.57
Attribution,numberFacets,0,0,0,0,0,100.0
Attribution,searchAttributes,17677926,0,190112,688781,18556819,96.0
Attribution,stringFacets,9490063,0,84486,321716,9896265,96.33
Caddyshack,assetSeoUrl,723752,2689,1129,0,727570,99.48
Caddyshack,catgroupSeq,20080,8165,7811727,34416844,42256816,0.08
Caddyshack,dsgCatgroups,8493509,0,505745,848716,9847970,87.55


Wrote field-level summary to `dev_sdsc_db`.`sdds_gold`.`field_level_comparison_summary`
Run 20260907_215400_996be16f complete in 7427.66s


field,MATCH,MISMATCH,total,similarity_pct,MISSING,EXTRA
assetSeoUrl,723752,2689,727570,99.48,1129,0
attributes,20856488,313991,25654587,81.17,409063,4075045
auxDescription2,0,0,0,100.0,0,0
buyable,727577,85,727662,99.99,0,0
caliaWebActive,0,0,14882,97.95,14882,0
catalogIds,931456,0,1722749,53.09,35129,756164
catentryId,0,727662,727662,0.0,0,0
catgroupSeq,20080,8165,42256816,0.08,7811727,34416844
color_family,722160,577,725951,99.48,2672,542
color_seq,399516,391,401344,99.75,1251,186
